# 🌸 Seha Personal AI — Kaggle Launcher
Run each cell from top to bottom. Add your own keys/numbers only in the configuration cell below. Do not publish this notebook after entering credentials.

In [ ]:
# 1) CONFIGURATION — edit these values
OWNER_NUMBER = ""       # Owner phone, country code + digits only, e.g. 9477XXXXXXX
WHATSAPP_NUMBER = ""    # Number to link with WhatsApp. Leave blank to use OWNER_NUMBER
OWNER_NAME = "Owner"
GEMINI_API_KEY = ""     # Paste Gemini API key here
GROQ_API_KEY = ""       # Optional fallback key; leave blank to disable fallback
GITHUB_OWNER = ""       # GitHub username/org; leave blank to disable cloud JSON sync
GITHUB_REPO = "Seha-Personal-AI"
GITHUB_TOKEN = ""       # Fine-grained token for this private repo: Contents read/write
MAX_MEDIA_MB = 15

assert OWNER_NUMBER, "Enter OWNER_NUMBER first."
if not WHATSAPP_NUMBER:
    WHATSAPP_NUMBER = OWNER_NUMBER
assert GEMINI_API_KEY or GROQ_API_KEY, "Enter at least one AI API key."
print("Configuration loaded. Keep this notebook private.")

In [ ]:
# 2) Install Node.js 22 and FFmpeg
import subprocess, os, sys
subprocess.run("curl -fsSL https://deb.nodesource.com/setup_22.x | bash -", shell=True, check=True)
subprocess.run("apt-get update -qq && apt-get install -y nodejs ffmpeg git -qq", shell=True, check=True)
print(subprocess.check_output(['node','--version'], text=True).strip())
print(subprocess.check_output(['ffmpeg','-version'], text=True).splitlines()[0])

In [ ]:
# 3) Clone your private repository (token is passed temporarily, not saved in Git config)
from pathlib import Path
import base64, shutil
PROJECT_DIR = Path('/kaggle/working/Seha-Personal-AI')
if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
if GITHUB_OWNER and GITHUB_TOKEN:
    auth = base64.b64encode(('x-access-token:' + GITHUB_TOKEN).encode()).decode()
    clone_url = f'https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git'
    subprocess.run(['git','-c',f'http.extraheader=AUTHORIZATION: basic {auth}','clone',clone_url,str(PROJECT_DIR)], check=True)
elif GITHUB_OWNER:
    clone_url = f'https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git'
    subprocess.run(['git','clone',clone_url,str(PROJECT_DIR)], check=True)
else:
    raise RuntimeError('Set GITHUB_OWNER and upload this project to GitHub first. The repository may be public if you do not use a token, but private is recommended.')
print('Project downloaded:', PROJECT_DIR)

In [ ]:
# 4) Install packages and verify source syntax
subprocess.run(['npm','install'], cwd=str(PROJECT_DIR), check=True)
subprocess.run(['npm','run','check'], cwd=str(PROJECT_DIR), check=True)
print('Dependencies installed and syntax checks passed.')

In [ ]:
# 5) Start Seha — pairing code appears in this cell output
import os
env = os.environ.copy()
env.update({
    'BOT_OWNER_NUMBER': OWNER_NUMBER,
    'WHATSAPP_NUMBER': WHATSAPP_NUMBER,
    'OWNER_NAME': OWNER_NAME,
    'GEMINI_API_KEY': GEMINI_API_KEY,
    'GROQ_API_KEY': GROQ_API_KEY,
    'GITHUB_OWNER': GITHUB_OWNER,
    'GITHUB_REPO': GITHUB_REPO,
    'GITHUB_TOKEN': GITHUB_TOKEN,
    'AUTH_DIR': '/kaggle/working/seha-auth',
    'MAX_MEDIA_MB': str(MAX_MEDIA_MB)
})
subprocess.run(['node','src/main.js'], cwd=str(PROJECT_DIR), env=env, check=False)

## Pair your phone
When the 8-digit code appears: WhatsApp → Settings → Linked Devices → Link a Device → **Link with phone number instead** → enter the code.

Kaggle sessions are temporary. WhatsApp auth files stay in `/kaggle/working/seha-auth` and are not uploaded to GitHub. If Kaggle resets that storage, pair again.